In [1]:
import numpy as np
import pandas as pd
import random
import csv
import geopandas as gpd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
import sys
import pickle

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *
from utils import process_greek

enc = 'utf-8'
shapefiles_folder = "C:/Users/dimit/Documents/noa hoard/Greece Shapefiles"

c:\Users\dimit\AppData\Local\Programs\Python\Python39\lib\site-packages\geopandas\_compat.py:112: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
NUTS0 = 'GR'
NUTS2 = 'Thessaly'

In [4]:
YEAR = 2023
MONTH = 'October'
PERIOD = '1st'

In [5]:
model = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Linear_model.pkl', 'rb'))
scaler = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Scaler.pkl', 'rb'))
imputer = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Imputer.pkl', 'rb'))

In [6]:
data_test = read_data(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Dataset_{YEAR}-{MONTH}-{PERIOD}.csv')
data_test.head()

,x,y,dt_placement,nuts2,lau1,day,month,week,year,day_sin,day_cos,month_sin,month_cos,week_sin,week_cos,area,population,population_density,eq_distance,ndvi,ndmi,ndwi,ndbi,ndvi_mean,ndmi_mean,ndwi_mean,ndbi_mean,ndvi_std,ndmi_std,ndwi_std,ndbi_std,lst,lst_day,lst_night,lst_jan_day_mean,lst_jan_night_mean,lst_feb_day_mean,lst_feb_night_mean,lst_mar_day_mean,lst_mar_night_mean,lst_apr_day_mean,lst_apr_night_mean,acc_rainfall_1week,acc_rainfall_2week,acc_rainfall_jan,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc,mosq_pred,mosq_previous,case
0,22.76105,39.69186,2023-10-01,θεσσαλιας,αγιας,1,10,39,2023,0.201299,0.97953,-0.866025,0.5,-0.99605,-0.088796,661.8,10705.0,17.3,45.75488,0.536489,0.220421,-0.429489,-0.220421,0.516155,0.231406,-0.409459,-0.231406,0.045120,0.051232,0.032706,0.051232,20.053750,22.286875,17.820625,10.178337,3.943677,10.285211,3.423839,16.579516,6.033934,17.971971,7.699233,167.369626,174.905063,1382.770841,8282.127022,1331.619869,1,123.585589,121.767859,183.036108,0.0,21.311709,31,90,30,90.0,30,90,10,10,1,6,6,2,0,31,0,0
1,22.72671,39.12560,2023-10-01,θεσσαλιας,αλμυρου,1,10,39,2023,0.201299,0.97953,-0.866025,0.5,-0.99605,-0.088796,905.4,16004.0,20.6,45.24728,0.381708,0.155278,-0.315646,-0.155278,0.348984,0.152418,-0.282003,-0.152418,0.042372,0.047149,0.032316,0.047149,19.330625,23.022500,15.638750,11.191823,3.758937,10.731281,3.006374,16.563729,6.106926,19.041462,7.924166,381.993820,400.626162,1874.428514,10017.207897,1610.666116,14,259.585198,342.571837,171.163634,0.0,3.415750,11,80,10,72.0,10,72,1,1,7,1,1,2,0,37,0,0
2,23.99842,39.24585,2023-10-01,θεσσαλιας,αλοννησου,1,10,39,2023,0.201299,0.97953,-0.866025,0.5,-0.99605,-0.088796,129.6,3153.0,21.2,46.00175,-0.172855,0.271654,0.345491,-0.271654,-0.168538,0.270502,0.342427,-0.270502,0.006998,0.003848,0.006507,0.003848,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.622663,9.752534,1639.419368,1484.683259,55302.082778,24,302.637680,165.397106,195.027810,0.0,1.676994,21,94,20,94.0,20,94,8,8,4,1,1,2,0,1,0,0
3,21.48510,39.29630,2023-10-01,θεσσαλιας,αργιθεας,1,10,39,2023,0.201299,0.97953,-0.866025,0.5,-0.99605,-0.088796,372.9,3515.0,9.3,44.78626,0.482337,0.251293,-0.390218,-0.251293,0.480886,0.231211,-0.385940,-0.231211,0.075676,0.080606,0.063765,0.080606,14.867692,17.677692,12.057692,6.988053,1.639104,7.062325,-0.605892,13.120876,3.493461,14.383348,4.274521,225.097578,229.684871,1478.282153,19327.278832,1925.318354,27,284.268894,1127.254453,234.678996,0.0,1.719393,21,88,20,88.0,20,88,8,8,4,1,1,2,0,47,0,0
4,22.93502,39.38117,2023-10-01,θεσσαλιας,βολου,1,10,39,2023,0.201299,0.97953,-0.866025,0.5,-0.99605,-0.088796,385.6,138865.0,374.6,45.57293,0.352396,0.117495,-0.285545,-0.117495,0.354177,0.099257,-0.295134,-0.099257,0.063323,0.065389,0.051166,0.065389,21.096667,23.520000,18.673333,12.043947,4.126457,11.394699,3.845437,17.241617,5.836050,19.956371,8.057589,373.063518,387.124510,2065.236841,5935.142744,3071.988689,5,143.181190,234.931813,174.426021,0.0,4.643743,31,91,30,91.0,30,91,10,10,1,6,6,2,0,42,0,0


In [7]:
X_test = data_test.select_dtypes(exclude=['object']).drop(columns = ['case'])
y_test = data_test['case']

X_test = scaler.transform(X_test)
X_test = imputer.fit_transform(X_test)

results_test = inference_lin_model(model, data_test, X_test, y_test)

In [8]:
results_test.drop(columns='case', inplace= True)
results_test

,x,y,lau1,day,month,year,score
0,22.15952,39.95174,ελασσονας,1,10,2023,0.155414
1,22.54288,39.85090,τεμπων,1,10,2023,0.129450
2,22.29611,39.76297,τυρναβου,1,10,2023,0.095243
3,22.34107,39.61048,λαρισαιων,1,10,2023,0.064102
4,21.77435,39.61293,τρικκαιων,1,10,2023,0.009546
5,22.51098,39.51390,κιλελερ,1,10,2023,0.008414
6,21.89535,39.26951,καρδιτσας,1,10,2023,0.008232
7,22.76105,39.69186,αγιας,1,10,2023,0.007855
8,22.02629,39.60300,φαρκαδονας,1,10,2023,0.006599
9,21.69908,39.40010,μουζακιου,1,10,2023,0.003468


In [9]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}.csv'

bins = []

with open(bins_path, mode='r', newline='') as file:
    reader = csv.reader(file)
    for row in reader:
        bins.extend(map(float, row))

print(bins)

[0.0, 0.0080309448997414, 0.0860446074909823, 0.5109550767195234, 0.8201371322355278, 0.9305335064919854, 1.0]


In [10]:
results_test['risk_class'] = pd.cut(results_test['score'], bins=bins, labels=False, include_lowest=True)
results_test

,x,y,lau1,day,month,year,score,risk_class
0,22.15952,39.95174,ελασσονας,1,10,2023,0.155414,2
1,22.54288,39.85090,τεμπων,1,10,2023,0.129450,2
2,22.29611,39.76297,τυρναβου,1,10,2023,0.095243,2
3,22.34107,39.61048,λαρισαιων,1,10,2023,0.064102,1
4,21.77435,39.61293,τρικκαιων,1,10,2023,0.009546,1
5,22.51098,39.51390,κιλελερ,1,10,2023,0.008414,1
6,21.89535,39.26951,καρδιτσας,1,10,2023,0.008232,1
7,22.76105,39.69186,αγιας,1,10,2023,0.007855,0
8,22.02629,39.60300,φαρκαδονας,1,10,2023,0.006599,0
9,21.69908,39.40010,μουζακιου,1,10,2023,0.003468,0


In [11]:
results_test.to_csv(f"../../data/{NUTS2}/results/{NUTS0}_{NUTS2}_Results_{YEAR}-{MONTH}-{PERIOD}.csv", encoding = enc, index = False)

In [12]:
##TODO Visualisation of results